In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from rasterstats import zonal_stats
from pathlib import Path

year = 2024
base = Path("/Users/dpro/projects/biomass/subprojects/mapping/ag_res")
raster_path = base / f"outputs/rasters/{year}/biomass_values_{year}.tif"
muni_path = base / "data/reference/municipalities.geojson"

# Load raster and get CRS and pixel size
with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    res_x, res_y = src.res
    pixel_area_km2 = abs(res_x * res_y) / 1e6

# Load municipality polygons and reproject to raster CRS
gdf = gpd.read_file(muni_path)
gdf = gdf.to_crs(raster_crs)

# Compute zonal stats
stats = zonal_stats(
    gdf,
    raster_path,
    stats=["mean", "sum", "count"],
    nodata=-9999.0,
    geojson_out=False
)

# Assemble DataFrame
df_stats = pd.DataFrame(stats)
df_stats["MUNI_NAME"] = gdf["MUNI_NAME"]
df_stats["area_km2"] = gdf.geometry.area / 1e6
df_stats["pixels_per_km2"] = df_stats["count"] / df_stats["area_km2"]
df_stats["total_biomass_tonnes"] = df_stats["sum"]
df_stats["mean_biomass_t_per_pixel"] = df_stats["mean"]

# Filter out empty ones
df_stats = df_stats[df_stats["count"] > 0]

# View summary
print(df_stats.describe()[["mean_biomass_t_per_pixel", "pixels_per_km2", "total_biomass_tonnes"]])
df_stats.sort_values('pixels_per_km2', ascending=False).head(10)


       mean_biomass_t_per_pixel  pixels_per_km2  total_biomass_tonnes
count                131.000000      131.000000            131.000000
mean                   0.066821      523.770648          31191.080543
std                    0.021136      309.441176          30716.035207
min                    0.017876        1.540906              2.352757
25%                    0.054652      259.710421           1184.848206
50%                    0.066595      508.124418          27932.164062
75%                    0.078391      774.267825          49793.261719
max                    0.161626     1048.987913         128645.562500


KeyError: 'pixel_area_km2'

In [4]:
# View summary
print(df_stats.describe()[["mean_biomass_t_per_pixel", "pixels_per_km2", "total_biomass_tonnes"]])
df_stats.sort_values('pixels_per_km2', ascending=False).head(10)

       mean_biomass_t_per_pixel  pixels_per_km2  total_biomass_tonnes
count                131.000000      131.000000            131.000000
mean                   0.066821      523.770648          31191.080543
std                    0.021136      309.441176          30716.035207
min                    0.017876        1.540906              2.352757
25%                    0.054652      259.710421           1184.848206
50%                    0.066595      508.124418          27932.164062
75%                    0.078391      774.267825          49793.261719
max                    0.161626     1048.987913         128645.562500


,mean,count,sum,MUNI_NAME,area_km2,pixels_per_km2,total_biomass_tonnes,mean_biomass_t_per_pixel
120,0.052658,2443024,128645.562500,MUNICIPALITY OF TWO BORDERS,2328.934366,1048.987913,128645.562500,0.052658
2,0.094352,508429,47971.312500,RM OF ROLAND,485.435755,1047.366196,47971.312500,0.094352
135,0.097964,998152,97782.734375,MUNICIPALITY OF RHINELAND,961.196767,1038.447105,97782.734375,0.097964
122,0.087241,1070701,93408.914062,RM OF MORRIS,1042.368190,1027.181192,93408.914062,0.087241
28,0.049963,1191362,59524.058594,RM OF PIPESTONE,1162.446439,1024.874747,59524.058594,0.049963
78,0.071528,794963,56861.917969,MUNICIPALITY OF BRENDA-WASKADA,778.545081,1021.087950,56861.917969,0.071528
63,0.085892,1172280,100689.117188,RM OF MACDONALD,1160.385058,1010.250858,100689.117188,0.085892
3,0.098020,471408,46207.187500,RM OF MONTCALM,471.919856,998.915375,46207.187500,0.098020
151,0.081175,538102,43680.210938,RM OF CARTIER,557.585807,965.056846,43680.210938,0.081175
114,0.096272,873325,84077.015625,RM OF DUFFERIN,919.008411,950.290541,84077.015625,0.096272


In [12]:
import geopandas as gpd
import rasterio
import pandas as pd
import numpy as np
from rasterstats import zonal_stats
from pathlib import Path

year = 2024
base = Path("/Users/dpro/projects/biomass/subprojects/mapping/ag_res")
raster_path = base / f"outputs/rasters/{year}/biomass_codes_{year}.tif"
muni_path = base / "data/reference/municipalities.geojson"
crop_ref_path = base / "data/reference/aci_crop_classifications_iac_classifications_des_cultures.csv"

# --- Load raster info ---
with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    res_x, res_y = src.res
    pixel_area_km2 = abs(res_x * res_y) / 1e6
    nodata_val = src.nodata
    all_codes = np.unique(src.read(1))
    all_codes = [int(c) for c in all_codes if c > 0]  # exclude nodata (0)
    print(f"Raster CRS: {raster_crs}")
    print(f"Valid crop codes: {len(all_codes)}")

# --- Load and reproject municipalities ---
gdf = gpd.read_file(muni_path).to_crs(raster_crs)
gdf["area_km2"] = gdf.geometry.area / 1e6

# --- Compute zonal stats for all crops in one pass ---
print("Computing zonal statistics for all crops...")
zs = zonal_stats(
    gdf,
    raster_path,
    categorical=True,
    nodata=nodata_val
)

# --- Convert results ---
records = []
for i, stats in enumerate(zs):
    muni = gdf.iloc[i]["MUNI_NAME"]
    for code, count in stats.items():
        if code == 0 or count == 0:
            continue
        records.append({
            "MUNI_NAME": muni,
            "Code": int(code),
            "pixel_count": int(count)
        })

df = pd.DataFrame(records)

# --- Merge area and compute density ---
df = df.merge(gdf[["MUNI_NAME", "area_km2"]], on="MUNI_NAME", how="left")
df["pixels_per_km2"] = df["pixel_count"] / df["area_km2"]

# --- Add crop label ---
crop_ref = pd.read_csv(crop_ref_path)
crop_ref["Code"] = crop_ref["Code"].astype(int)
crop_ref["Label"] = crop_ref["Label"].astype(str).str.strip()
df = df.merge(crop_ref[["Code", "Label"]], on="Code", how="left")

# --- Clean and order ---
df = df[["MUNI_NAME", "Label", "Code", "pixel_count", "pixels_per_km2", "area_km2"]]
df = df[df["pixel_count"] > 0].reset_index(drop=True)

# --- Save output for reuse ---
out_path = base / f"outputs/reports/raster_build/pixel_density_by_crop_{year}.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f"Saved detailed pixel-density report → {out_path}")

# --- Quick summary ---
print("Top 15 crops by average pixel density:")
print(df.groupby("Label")["pixels_per_km2"].mean().sort_values(ascending=False).head(15))



Raster CRS: PROJCS["unnamed",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",40],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",44.75],PARAMETER["standard_parallel_2",55.75],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Valid crop codes: 41
Computing zonal statistics for all crops...
Saved detailed pixel-density report → /Users/dpro/projects/biomass/subprojects/mapping/ag_res/outputs/reports/raster_build/pixel_density_by_crop_2024.csv
Top 15 crops by average pixel density:
Label
Canola/rapeseed        165.914818
Spring wheat           160.249736
Urban/developed        150.982317
Broadleaf              137.635404
Grassl

In [19]:
import pandas as pd

target_rms = [
    "RM OF ELLICE-ARCHIE",
    "RM OF WALLACE-WOODWORTH",
    "MUNICIPALITY OF TWO BORDERS",
    "RM OF PIPESTONE",
    "PRAIRIE VIEW MUNICIPALITY"
]

# Compute provincial baseline (excluding all target RMs)
baseline = (
    df[~df["MUNI_NAME"].isin(target_rms)]
    .groupby("Label")[["pixels_per_km2", "pixel_count"]]
    .mean()
    .rename(columns=lambda x: f"{x}_others")
)

# Gather per-RM summaries in one vertical DataFrame
records = []

for rm in target_rms:
    sub = df[df["MUNI_NAME"] == rm]
    rm_summary = (
        sub.groupby("Label")[["pixels_per_km2", "pixel_count"]]
        .mean()
        .join(baseline, how="outer")
    )

    rm_summary["density_ratio"] = rm_summary["pixels_per_km2"] / rm_summary["pixels_per_km2_others"]
    rm_summary["MUNI_NAME"] = rm
    records.append(rm_summary.reset_index())

compare_long = pd.concat(records, ignore_index=True)

# Clean up display
compare_long = compare_long[
    ["MUNI_NAME", "Label", "pixels_per_km2", "pixels_per_km2_others", "density_ratio", "pixel_count"]
]

compare_long.sort_values(["pixels_per_km2"], ascending=[False]).head(30)


,MUNI_NAME,Label,pixels_per_km2,pixels_per_km2_others,density_ratio,pixel_count
114,MUNICIPALITY OF TWO BORDERS,Spring wheat,377.596730,153.584264,2.458564,879398.0
155,RM OF PIPESTONE,Spring wheat,349.180819,153.584264,2.273546,405904.0
170,PRAIRIE VIEW MUNICIPALITY,Canola/rapeseed,329.326222,161.236994,2.042498,573709.0
129,RM OF PIPESTONE,Canola/rapeseed,327.861127,161.236994,2.033411,381121.0
73,RM OF WALLACE-WOODWORTH,Spring wheat,311.843337,153.584264,2.030438,631708.0
196,PRAIRIE VIEW MUNICIPALITY,Spring wheat,306.102687,153.584264,1.993060,533252.0
32,RM OF ELLICE-ARCHIE,Spring wheat,283.043674,153.584264,1.842921,328346.0
88,MUNICIPALITY OF TWO BORDERS,Canola/rapeseed,267.044022,161.236994,1.656221,621928.0
47,RM OF WALLACE-WOODWORTH,Canola/rapeseed,264.531324,161.236994,1.640637,535867.0
25,RM OF ELLICE-ARCHIE,Pasture/forages,257.747442,26.027803,9.902774,299001.0


In [20]:
import rasterio
import numpy as np
from rasterio.windows import Window
from scipy.ndimage import uniform_filter
from pathlib import Path
import math

# --- Parameters ---
year = 2024
window_pixels = 33       # e.g. ~1 km neighbourhood
tile_size = 2048         # number of pixels per tile (square)
base = Path("/Users/dpro/projects/biomass/subprojects/mapping/ag_res")
in_path = base / f"outputs/rasters/{year}/biomass_values_{year}.tif"
out_path = base / f"outputs/rasters/{year}/biomass_local_density_t_per_km2_{year}_chunked.tif"

# --- Open source raster and prepare output ---
with rasterio.open(in_path) as src:
    profile = src.profile.copy()
    nodata = src.nodata
    res_x, res_y = src.res
    pixel_area_m2 = abs(res_x * res_y)
    pixel_area_km2 = pixel_area_m2 / 1_000_000.0

    profile.update(dtype="float32", nodata=nodata, compress="LZW", predictor=2)

    height, width = src.height, src.width
    overlap = window_pixels  # pad this many pixels around each tile

    with rasterio.open(out_path, "w", **profile) as dst:

        # iterate over tiles
        n_tiles_x = math.ceil(width / tile_size)
        n_tiles_y = math.ceil(height / tile_size)

        for ty in range(n_tiles_y):
            for tx in range(n_tiles_x):
                x0 = tx * tile_size
                y0 = ty * tile_size
                w = min(tile_size, width - x0)
                h = min(tile_size, height - y0)

                # read with overlap
                x_read = max(0, x0 - overlap)
                y_read = max(0, y0 - overlap)
                w_read = min(tile_size + 2 * overlap, width - x_read)
                h_read = min(tile_size + 2 * overlap, height - y_read)
                window_read = Window(x_read, y_read, w_read, h_read)

                arr = src.read(1, window=window_read).astype("float32")
                valid = (arr != nodata) & ~np.isnan(arr)

                biomass_filled = np.where(valid, arr, 0.0).astype("float32")
                valid_filled = valid.astype("float32")

                biomass_mean = uniform_filter(biomass_filled, size=window_pixels, mode="constant", cval=0.0)
                valid_mean = uniform_filter(valid_filled, size=window_pixels, mode="constant", cval=0.0)

                window_area_pixels = window_pixels * window_pixels
                biomass_sum = biomass_mean * window_area_pixels
                valid_sum = valid_mean * window_area_pixels
                window_area_km2 = valid_sum * pixel_area_km2

                with np.errstate(divide="ignore", invalid="ignore"):
                    local_density = biomass_sum / window_area_km2
                local_density[valid_sum == 0] = nodata

                # crop back to the inner tile region (remove overlap edges)
                x_inner = overlap if x_read > 0 else 0
                y_inner = overlap if y_read > 0 else 0
                x_end = x_inner + w
                y_end = y_inner + h
                tile_data = local_density[y_inner:y_end, x_inner:x_end]

                dst.write(tile_data.astype("float32"), 1, window=Window(x0, y0, w, h))

            print(f"Row tile {ty+1}/{n_tiles_y} complete")

print(f"Wrote chunked local density raster: {out_path}")


Row tile 1/12 complete
Row tile 2/12 complete
Row tile 3/12 complete
Row tile 4/12 complete
Row tile 5/12 complete
Row tile 6/12 complete
Row tile 7/12 complete
Row tile 8/12 complete
Row tile 9/12 complete
Row tile 10/12 complete
Row tile 11/12 complete
Row tile 12/12 complete
Wrote chunked local density raster: /Users/dpro/projects/biomass/subprojects/mapping/ag_res/outputs/rasters/2024/biomass_local_density_t_per_km2_2024_chunked.tif
